# EDA: динамика частотности по трём фразам (issue #62)

Разведочный анализ фикстур `tests/fixtures/dynamics_*.csv` — месячные ряды
Яндекс Вордстат за 24 месяца (август 2024 — июль 2026), Россия, все устройства:

- **высокочастотная** — «купить телефон» (`dynamics_high_freq.csv`);
- **среднечастотная** — «курсы английского языка» (`dynamics_mid_freq.csv`);
- **сезонная** — «новогодние подарки» (`dynamics_seasonal.csv`).

Чтение — строго через `src/wordstat_trends/loader.py` (`load_run`): вокруг
каждой фикстуры собирается минимальный run-каталог (`manifest.json` +
`dynamics.csv`), как это делает `tests/test_loader.py` для реальных прогонов
`wordstat collect --keep-raw`.

Выводы — текстом в конце ноутбука; они вход для карточки датасета (#63) и
выбора метрик.

In [1]:
import sys
import tempfile
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from wordstat.models import CollectionManifest, ExportSummary, WordstatView

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from wordstat_trends.loader import load_run  # noqa: E402

FIXTURES = REPO / "tests" / "fixtures"
PHRASES = {
    "dynamics_high_freq.csv": "купить телефон",
    "dynamics_mid_freq.csv": "курсы английского языка",
    "dynamics_seasonal.csv": "новогодние подарки",
}

In [2]:
def make_run(base: Path, csv_name: str, phrase: str) -> Path:
    """Минимальный run-каталог поверх фикстуры (зеркалит tests/test_loader.py)."""
    run = base / csv_name.removesuffix(".csv")
    run.mkdir(parents=True)
    (run / "dynamics.csv").write_bytes((FIXTURES / csv_name).read_bytes())
    manifest = CollectionManifest(
        phrase=phrase,
        region="Россия",
        created_at=datetime(2026, 8, 21, 1, 40, 38, tzinfo=UTC),
        source_url="https://wordstat.yandex.ru/",
        exports=[
            ExportSummary(
                view=WordstatView.DYNAMICS,
                file="dynamics.parquet",
                raw_file="dynamics.csv",
                row_count=24,
                dtypes={"Период": "string", "Число запросов": "int64"},
            )
        ],
    )
    (run / "manifest.json").write_text(manifest.model_dump_json(indent=2), encoding="utf-8")
    return run


tmp = tempfile.mkdtemp(prefix="wordstat-eda-")
frames = []
for csv_name, phrase in PHRASES.items():
    df = load_run(make_run(Path(tmp), csv_name, phrase))
    assert df.attrs["phrase"] == phrase
    frames.append(df.assign(phrase=phrase))

data = pd.concat(frames, ignore_index=True)
data["period"] = pd.PeriodIndex(data["period"], freq="M").to_timestamp()
data

,period,queries,share_pct,phrase
0,2024-08-01,997977,0.01150,купить телефон
1,2024-09-01,924579,0.01000,купить телефон
2,2024-10-01,923940,0.00880,купить телефон
3,2024-11-01,942229,0.00870,купить телефон
4,2024-12-01,1022068,0.00910,купить телефон
...,...,...,...,...
67,2026-03-01,30964,0.00029,новогодние подарки
68,2026-04-01,24552,0.00023,новогодние подарки
69,2026-05-01,22569,0.00022,новогодние подарки
70,2026-06-01,18998,0.00019,новогодние подарки


## 1. Длина рядов, непрерывность, пропуски

In [3]:
summary = []
for phrase, group in data.groupby("phrase", sort=False):
    periods = pd.PeriodIndex(group["period"], freq="M")
    full_range = pd.period_range(periods.min(), periods.max(), freq="M")
    summary.append(
        {
            "фраза": phrase,
            "строк": len(group),
            "период": f"{periods.min()} — {periods.max()}",
            "месяцев в диапазоне": len(full_range),
            "пропусков месяцев": len(full_range) - len(periods),
            "NaN в queries/share": int(group[["queries", "share_pct"]].isna().sum().sum()),
            "дубликатов периода": int(periods.duplicated().sum()),
        }
    )
pd.DataFrame(summary)

,фраза,строк,период,месяцев в диапазоне,пропусков месяцев,NaN в queries/share,дубликатов периода
0,купить телефон,24,2024-08 — 2026-07,24,0,0,0
1,курсы английского языка,24,2024-08 — 2026-07,24,0,0,0
2,новогодние подарки,24,2024-08 — 2026-07,24,0,0,0


## 2. Распределения частотностей

In [4]:
dist = []
for phrase, group in data.groupby("phrase", sort=False):
    q = group["queries"]
    s = group["share_pct"]
    dist.append(
        {
            "фраза": phrase,
            "медиана запросов": int(q.median()),
            "min запросов": int(q.min()),
            "max запросов": int(q.max()),
            "max/min": round(q.max() / q.min(), 1),
            "CV queries": round(q.std() / q.mean(), 2),
            "доля % медиана": round(s.median(), 5),
            "доля % min—max": f"{s.min():.5f} — {s.max():.5f}",
            "CV share": round(s.std() / s.mean(), 2),
        }
    )
pd.DataFrame(dist)

,фраза,медиана запросов,min запросов,max запросов,max/min,CV queries,доля % медиана,доля % min—max,CV share
0,купить телефон,854705,643202,1228504,1.9,0.15,0.00815,0.00650 — 0.01160,0.16
1,курсы английского языка,58495,40954,95108,2.3,0.24,0.00055,0.00038 — 0.00092,0.25
2,новогодние подарки,47286,18998,1234632,65.0,1.67,0.00048,0.00019 — 0.01106,1.63


In [5]:
data.pivot(index="period", columns="phrase", values="queries").describe().astype("int64")

phrase,купить телефон,курсы английского языка,новогодние подарки
count,24,24,24
mean,862068,57584,199576
std,129171,13570,333177
min,643202,40954,18998
25%,755303,45265,30680
50%,854705,58495,47286
75%,925934,64716,183025
max,1228504,95108,1234632


In [6]:
# Квартальная динамика (млн запросов/мес, среднее по кварталу) — тренд без месячного шума.
(
    data.assign(quarter=data["period"].dt.to_period("Q"))
    .groupby(["quarter", "phrase"], sort=False)["queries"].mean()
    .unstack("phrase").div(1e6).round(2)
)

phrase,купить телефон,курсы английского языка,новогодние подарки
quarter,,,
2024Q3,0.96,0.07,0.06
2024Q4,0.96,0.06,0.65
2025Q1,0.88,0.06,0.09
2025Q2,0.76,0.06,0.03
2025Q3,0.88,0.07,0.06
2025Q4,0.87,0.06,0.61
2026Q1,0.70,0.04,0.08
2026Q2,0.79,0.04,0.02
2026Q3,1.23,0.04,0.03


## 3. Выбросы

In [7]:
outliers = []
for phrase, group in data.groupby("phrase", sort=False):
    q = group.set_index("period")["queries"]
    med = q.median()
    peak_month = q.idxmax()
    outliers.append(
        {
            "фраза": phrase,
            "пик": f"{peak_month:%Y-%m} = {q.max():,}".replace(",", " "),
            "×от медианы": round(q.max() / med, 1),
            "×от минимума": round(q.max() / q.min(), 1),
            "минимум": f"{q.idxmin():%Y-%m} = {q.min():,}".replace(",", " "),
            "вне 1.5·IQR": int(((q < q.quantile(0.25) - 1.5 * (q.quantile(0.75) - q.quantile(0.25)))
                                 | (q > q.quantile(0.75) + 1.5 * (q.quantile(0.75) - q.quantile(0.25)))).sum()),
        }
    )
pd.DataFrame(outliers)

,фраза,пик,×от медианы,×от минимума,минимум,вне 1.5·IQR
0,купить телефон,2026-07 = 1 228 504,1.4,1.9,2026-02 = 643 202,1
1,курсы английского языка,2025-09 = 95 108,1.6,2.3,2026-07 = 40 954,1
2,новогодние подарки,2024-12 = 1 234 632,26.1,65.0,2026-06 = 18 998,4


In [8]:
# Сезонная фраза: ранжирование месяцев по медианному числу запросов — форма годового цикла.
seasonal = data[data["phrase"] == "новогодние подарки"]
seasonal_by_month = seasonal.assign(month=seasonal["period"].dt.month).groupby("month")["queries"].median()
seasonal_by_month.sort_values(ascending=False).astype("int64").to_frame("медиана запросов по месяцу")

,медиана запросов по месяцу
month,
12,1192819
11,484094
10,220807
1,167994
9,97161
2,48297
8,42565
3,34634
7,31821


In [9]:
# Согласованность двух лет сезонной фразы: корреляция month-aligned (по значениям,
# не по индексу — годы не пересекаются, Series.corr после align дал бы NaN).
p = seasonal.set_index("period")["queries"]
y1 = p["2024-08":"2025-07"].to_numpy()
y2 = p["2025-08":"2026-07"].to_numpy()
round(pd.Series(y1).corr(pd.Series(y2)), 3)

np.float64(0.999)

## Выводы (вход для карточки датасета #63 и выбора метрик)

1. **Структура.** Три ряда по 24 месяца (2024-08 — 2026-07), непрерывные, без
   пропусков месяцев, без NaN и дубликатов периодов. Колонки загрузчика:
   `period` (`YYYY-MM`), `queries` (int64), `share_pct` (float, доля %).
   Источник — `load_run()` из `wordstat_trends.loader`; формат сырых CSV
   (UTF-8 BOM, CR-only, `;`, десятичная запятая, тысячи через пробел) парсится
   загрузчиком без потерь.
2. **Три режима частотности.** Медианы: «купить телефон» ~855 тыс.
   запросов/мес (доля 0.65–1.16%), «курсы английского языка» ~58 тыс.
   (41–95 тыс.), «новогодние подарки» ~47 тыс. Медианы различаются в ~18 раз
   → абсолютные значения запросов несравнимы между фразами, метрики нужны
   масштабо-инвариантные: относительные ошибки (sMAPE/WAPE/MASE), а не
   MAE/RMSE.
3. **Волатильность.** `share_pct` повторяет форму `queries` (CV обеих колонок
   совпадает с точностью до сотых): доля — та же частотность, нормированная
   на объём всех запросов, для внутрифразовой динамики эквивалентна числу
   запросов. CV числа запросов: 0.15 (высокочастотная), 0.24
   (среднечастотная), 1.67 (сезонная).
4. **Тренд.** Высокочастотная фраза проседает к весне 2026 (квартальные
   средние 0.96 млн в 2024Q3 → 0.70 млн в 2026Q1) и даёт летний пик
   2026-07 (~1.23 млн) — ряд «почти стационарный с лёгким сезонным горбом»,
   max/min всего 1.9. Сезонная фраза — чистый годовой цикл без тренда: два
   года почти идеально совпадают по месяцам (month-aligned корреляция 0.999).
5. **Выбросы — это сезонность, а не шум.** Декабрьский пик «новогодних
   подарков» (дек 2024: 1 234 632) — ×65 от минимума (июнь 2026: 18 998) и
   ×26 от медианы; формальный тест 1.5·IQR помечает 4 точки (два декабря и
   соседние ноябри) как выбросы. Но это воспроизводимый годовой максимум
   (два декабря подряд: 1.23 млн и 1.15 млн), поэтому «чистить» их нельзя —
   они и есть сигнал. У высоко- и среднечастотной фраз вне 1.5·IQR по одной
   точке — пик, а не ошибка данных.
6. **Следствие для метрик и моделей.** (a) Ошибку считать пофразно и в
   относительной шкале; сезонную фразу оценивать отдельно от
   высокочастотной — на ×65-амплитуде обязателен sMAPE, иначе декабрь
   доминирует в MAE. (b) Порог «аномалия» определять относительно
   сезонного профиля (month-aligned), а не глобальных квантилей ряда.
   (c) 24 точки — короткий ряд: валидация только скользящим окном
   (rolling origin), holdout из последних месяцев; два полных годовых
   цикла дают минимум для оценки сезонной компоненты (sp=12).
7. **Ограничения.** По одной фразе на «режим» — обобщать распределение
   частотностей на весь корпус нельзя; день/неделя не покрыты (есть только
   отдельные daily-фикстуры `*_daily_*`, здесь не анализируются).